# Understanding Confidence Intervals

This notebook builds intuition for confidence intervals from the ground up — what they are, what they are *not*, and how to use them correctly.

## Table of Contents
1. [The Core Idea](#core-idea)
2. [Sampling Distributions](#sampling)
3. [Building a Confidence Interval](#building)
4. [The Frequentist Interpretation](#interpretation)
5. [Effect of Sample Size and Confidence Level](#effects)
6. [Common Misconceptions](#misconceptions)
7. [Real-World Example: A/B Testing](#ab-testing)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

# Style
sns.set_theme(style='whitegrid', palette='muted')
rng = np.random.default_rng(42)

print('Libraries loaded.')

---
## 1. The Core Idea <a id='core-idea'></a>

Imagine you want to know the **true mean** of some population (e.g. average height of all adults in a country). You can't measure everyone, so you take a **sample** and compute a sample mean $\bar{x}$.

The problem: $\bar{x}$ is just an estimate. A different sample would give a different $\bar{x}$. A confidence interval (CI) quantifies this uncertainty by giving a **range** that likely contains the true mean $\mu$.

### Setup: A population we can inspect
For pedagogical purposes, let's define a population whose true mean we *know*, then see how well CIs capture it.

In [ ]:
# True population: adult heights (cm), normally distributed
TRUE_MEAN = 170
TRUE_STD  = 10
POPULATION_SIZE = 100_000

population = rng.normal(TRUE_MEAN, TRUE_STD, POPULATION_SIZE)

fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(population, bins=80, kde=True, ax=ax, color='steelblue', alpha=0.6)
ax.axvline(TRUE_MEAN, color='crimson', lw=2.5, label=f'True mean μ = {TRUE_MEAN} cm')
ax.set_xlabel('Height (cm)')
ax.set_ylabel('Count')
ax.set_title('Our Population (we normally cannot see this)')
ax.legend()
plt.tight_layout()
plt.show()

---
## 2. Sampling Distributions <a id='sampling'></a>

If we draw many samples of size $n$ and record each sample mean $\bar{x}$, those means form a **sampling distribution**.

The **Central Limit Theorem** tells us:
$$\bar{x} \sim \mathcal{N}\!\left(\mu,\; \frac{\sigma}{\sqrt{n}}\right)$$

The standard deviation of this distribution — $\sigma/\sqrt{n}$ — is the **Standard Error (SE)**.

In [ ]:
N_SAMPLES = 2000
sample_sizes = [5, 20, 100]

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

for ax, n in zip(axes, sample_sizes):
    sample_means = [rng.choice(population, n).mean() for _ in range(N_SAMPLES)]
    se = TRUE_STD / np.sqrt(n)

    sns.histplot(sample_means, bins=50, kde=True, ax=ax, color='steelblue', alpha=0.6)
    ax.axvline(TRUE_MEAN, color='crimson', lw=2, linestyle='--', label=f'μ = {TRUE_MEAN}')
    ax.set_title(f'n = {n}\nSE = σ/√n = {se:.2f} cm')
    ax.set_xlabel('Sample mean (cm)')
    ax.legend(fontsize=8)

axes[0].set_ylabel('Count')
fig.suptitle('Sampling Distribution of the Mean — more data → tighter spread', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

**Key takeaway:** The sampling distribution narrows as $n$ grows. Bigger samples → more precise estimates.

---
## 3. Building a Confidence Interval <a id='building'></a>

For a 95% CI using a known $\sigma$ (or large $n$ so we can use $z$):

$$\text{CI} = \bar{x} \pm z^* \cdot \frac{\sigma}{\sqrt{n}}$$

where $z^* = 1.96$ for 95% confidence (the middle 95% of the standard normal).

In practice with unknown $\sigma$, we use the **t-distribution** with $n-1$ degrees of freedom:

$$\text{CI} = \bar{x} \pm t^*_{n-1} \cdot \frac{s}{\sqrt{n}}$$

In [ ]:
# Visualise where z* = 1.96 comes from
fig, ax = plt.subplots(figsize=(9, 4))

x = np.linspace(-4, 4, 400)
y = stats.norm.pdf(x)

ax.plot(x, y, 'steelblue', lw=2.5)

# Shade middle 95%
z_star = 1.96
x_fill = np.linspace(-z_star, z_star, 300)
ax.fill_between(x_fill, stats.norm.pdf(x_fill), alpha=0.35, color='steelblue', label='Middle 95%')

# Shade tails
for side in [-1, 1]:
    x_tail = np.linspace(side * 4, side * z_star, 100)
    ax.fill_between(x_tail, stats.norm.pdf(x_tail), alpha=0.5, color='salmon', label='2.5% tail' if side == -1 else '')

ax.axvline(-z_star, color='crimson', lw=1.5, linestyle='--')
ax.axvline( z_star, color='crimson', lw=1.5, linestyle='--')
ax.text(-z_star - 0.1, 0.22, f'−{z_star}', ha='right', color='crimson', fontsize=11)
ax.text( z_star + 0.1, 0.22, f'+{z_star}', ha='left',  color='crimson', fontsize=11)
ax.annotate('', xy=(z_star, 0.15), xytext=(-z_star, 0.15),
            arrowprops=dict(arrowstyle='<->', color='steelblue', lw=2))
ax.text(0, 0.17, '95% of area', ha='center', color='steelblue', fontsize=11)

ax.set_xlabel('Standard deviations from mean (z)')
ax.set_ylabel('Probability density')
ax.set_title('Standard Normal: z* = 1.96 captures the middle 95%')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Compute a single 95% CI from one sample
sample = rng.choice(population, size=30)
n = len(sample)
x_bar = sample.mean()
s = sample.std(ddof=1)
se = s / np.sqrt(n)

confidence = 0.95
t_star = stats.t.ppf((1 + confidence) / 2, df=n - 1)
margin = t_star * se

ci_low  = x_bar - margin
ci_high = x_bar + margin

print(f'Sample mean  : {x_bar:.2f} cm')
print(f'Sample std   : {s:.2f} cm')
print(f'Standard error: {se:.2f} cm')
print(f't* (df={n-1}): {t_star:.3f}')
print(f'Margin of error: ±{margin:.2f} cm')
print(f'95% CI       : [{ci_low:.2f}, {ci_high:.2f}]')
print(f'True mean μ  : {TRUE_MEAN} cm  →  captured? {ci_low <= TRUE_MEAN <= ci_high}')

---
## 4. The Frequentist Interpretation <a id='interpretation'></a>

> **"If we repeat this procedure many times, 95% of the resulting intervals will contain the true mean."**

The confidence level refers to the **procedure**, not to any single interval. Let's make this concrete by drawing 100 different samples and plotting their CIs.

In [ ]:
N_INTERVALS = 100
n = 30
confidence = 0.95
t_star = stats.t.ppf((1 + confidence) / 2, df=n - 1)

intervals = []
for _ in range(N_INTERVALS):
    s = rng.choice(population, size=n)
    xb = s.mean()
    se = s.std(ddof=1) / np.sqrt(n)
    lo, hi = xb - t_star * se, xb + t_star * se
    intervals.append((xb, lo, hi, lo <= TRUE_MEAN <= hi))

captured = sum(r[3] for r in intervals)

fig, ax = plt.subplots(figsize=(10, 9))

for i, (xb, lo, hi, hit) in enumerate(intervals):
    color = 'steelblue' if hit else 'crimson'
    ax.plot([lo, hi], [i, i], color=color, lw=1.2, alpha=0.8)
    ax.plot(xb, i, 'o', color=color, ms=3)

ax.axvline(TRUE_MEAN, color='black', lw=2, linestyle='--', label=f'True μ = {TRUE_MEAN}')

blue_patch = mpatches.Patch(color='steelblue', label=f'Contains μ ({captured}/{N_INTERVALS})')
red_patch  = mpatches.Patch(color='crimson',   label=f'Misses μ ({N_INTERVALS - captured}/{N_INTERVALS})')
ax.legend(handles=[blue_patch, red_patch, plt.Line2D([0], [0], color='black', lw=2, linestyle='--', label=f'True μ = {TRUE_MEAN}')],
          loc='lower right')

ax.set_xlabel('Height (cm)')
ax.set_ylabel('Sample index')
ax.set_title(f'100 independent 95% CIs (n={n})\n{captured}% contain the true mean')
plt.tight_layout()
plt.show()

The red intervals are the ones that missed. Notice:
- Roughly 5 out of 100 miss — exactly what we'd expect.
- There's nothing special about any individual red interval; it just got unlucky.
- You **cannot** say "there's a 95% chance μ is in *this* interval" — μ is fixed, the interval is random.

---
## 5. Effect of Sample Size and Confidence Level <a id='effects'></a>

Two dials control CI width:
- **Larger $n$** → narrower CI (more information)
- **Higher confidence level** → wider CI (you need a bigger net to be more sure you catch the fish)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: vary sample size ---
ax = axes[0]
ns = [5, 10, 20, 50, 100, 200, 500]
widths_n = []
for n in ns:
    samp = rng.choice(population, size=n)
    se = samp.std(ddof=1) / np.sqrt(n)
    t_star = stats.t.ppf(0.975, df=n - 1)
    widths_n.append(2 * t_star * se)

sns.lineplot(x=ns, y=widths_n, ax=ax, marker='o', color='steelblue')
ax.set_xlabel('Sample size (n)')
ax.set_ylabel('CI width (cm)')
ax.set_title('95% CI width vs Sample Size\n(decreases as ~1/√n)')

# Overlay theoretical curve
n_range = np.linspace(5, 500, 200)
theoretical = 2 * 1.96 * TRUE_STD / np.sqrt(n_range)
ax.plot(n_range, theoretical, 'r--', alpha=0.6, label='Theoretical (known σ)')
ax.legend()

# --- Right: vary confidence level ---
ax = axes[1]
confidence_levels = [0.50, 0.80, 0.90, 0.95, 0.99, 0.999]
samp = rng.choice(population, size=50)
xb = samp.mean()
se = samp.std(ddof=1) / np.sqrt(50)

for i, cl in enumerate(confidence_levels):
    t_star = stats.t.ppf((1 + cl) / 2, df=49)
    lo, hi = xb - t_star * se, xb + t_star * se
    color = plt.cm.Blues(0.3 + 0.7 * (i / (len(confidence_levels) - 1)))
    ax.barh(f'{int(cl*100)}%', hi - lo, left=lo, height=0.5, color=color, alpha=0.85)
    ax.text(xb, i, f'±{t_star*se:.1f}', va='center', ha='center', fontsize=8, color='white', fontweight='bold')

ax.axvline(TRUE_MEAN, color='crimson', lw=2, linestyle='--', label=f'True μ = {TRUE_MEAN}')
ax.axvline(xb,        color='black',   lw=1.5, linestyle=':',  label=f'x̄ = {xb:.1f}')
ax.set_xlabel('Height (cm)')
ax.set_title('CI width vs Confidence Level (n=50)\n(higher confidence → wider interval)')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

**Trade-off summary:**

| Want narrower CI? | Want higher confidence? |
|---|---|
| Collect more data (↑ n) | Accept a wider interval |
| Accept lower confidence level | Collect more data (↑ n) |

You cannot get a very narrow CI *and* very high confidence from a small sample — that's the fundamental constraint of statistical estimation.

---
## 6. Common Misconceptions <a id='misconceptions'></a>

Let's visualise the most common wrong interpretations.

In [ ]:
# Generate one CI to use in the misconception discussion
samp = rng.choice(population, size=40)
n = len(samp)
xb = samp.mean()
se = samp.std(ddof=1) / np.sqrt(n)
t_star = stats.t.ppf(0.975, df=n - 1)
lo, hi = xb - t_star * se, xb + t_star * se

misconceptions = [
    {
        'label': '❌ Wrong: "95% of individual\n    observations fall in this range"',
        'color': 'salmon',
        'range': (lo, hi),
        'note': f'Individual observations span\n[{xb-2*TRUE_STD:.0f}, {xb+2*TRUE_STD:.0f}] (±2σ), much wider!'
    },
    {
        'label': '❌ Wrong: "μ has a 95% prob.\n    of being in this range"',
        'color': 'lightsalmon',
        'range': (lo, hi),
        'note': 'μ is fixed. It either is or\nisn\'t in the interval (we just don\'t know).'
    },
    {
        'label': '✓ Correct: "This procedure produces\n    intervals that contain μ 95% of the time"',
        'color': 'steelblue',
        'range': (lo, hi),
        'note': 'Statement about the long-run\nbehaviour of the procedure.'
    },
]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, m in zip(axes, misconceptions):
    # For the first panel, also show individual data spread
    if '❌' in m['label'] and 'observations' in m['label']:
        sns.kdeplot(samp, ax=ax, color='grey', fill=True, alpha=0.15, label='Sample KDE')
        obs_lo, obs_hi = xb - 2 * TRUE_STD, xb + 2 * TRUE_STD
        ax.axvspan(obs_lo, obs_hi, alpha=0.12, color='orange', label='±2σ (95% of obs.)')

    lo_, hi_ = m['range']
    ax.axvspan(lo_, hi_, alpha=0.35, color=m['color'])
    ax.axvline(lo_, color=m['color'], lw=2)
    ax.axvline(hi_, color=m['color'], lw=2)
    ax.axvline(xb,        color='black',   lw=1.5, linestyle=':',  label=f'x̄={xb:.1f}')
    ax.axvline(TRUE_MEAN, color='crimson', lw=2,   linestyle='--', label=f'μ={TRUE_MEAN}')

    ax.set_title(m['label'], fontsize=10, loc='left')
    ax.set_xlabel('Height (cm)')
    ax.text(0.5, 0.05, m['note'], transform=ax.transAxes,
            ha='center', va='bottom', fontsize=8.5,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
    ax.legend(fontsize=7.5)
    ax.set_xlim(140, 200)

fig.suptitle('Confidence Interval Misconceptions vs Correct Interpretation', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

### Summary of misconceptions

| Misconception | Why it's wrong |
|---|---|
| "95% of data points fall within the CI" | That's a **prediction interval**, which is much wider (uses σ, not SE) |
| "There's a 95% chance μ is in this CI" | μ is a fixed constant — it doesn't have probabilities. The *procedure* has the 95% property |
| "If two CIs don't overlap, the means are significantly different" | Overlapping CIs don't directly map to significance tests; use a proper t-test |

---
## 7. Real-World Example: A/B Testing <a id='ab-testing'></a>

A classic use case: you run two variants of a webpage and measure conversion rates. CIs help you decide if the difference is real.

In [ ]:
# Simulate A/B test: conversion rate (0 or 1 per visitor)
true_rate_A = 0.10   # 10% conversion
true_rate_B = 0.125  # 12.5% conversion — a real effect

n_A, n_B = 500, 500

data_A = rng.binomial(1, true_rate_A, n_A)
data_B = rng.binomial(1, true_rate_B, n_B)

# Wilson score interval for proportions (more accurate for small counts)
def wilson_ci(successes, n, confidence=0.95):
    z = stats.norm.ppf((1 + confidence) / 2)
    p_hat = successes / n
    centre = (p_hat + z**2 / (2 * n)) / (1 + z**2 / n)
    half_width = z * np.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centre - half_width, centre + half_width

p_A = data_A.mean()
p_B = data_B.mean()
ci_A = wilson_ci(data_A.sum(), n_A)
ci_B = wilson_ci(data_B.sum(), n_B)

# CI on the difference (normal approx)
diff = p_B - p_A
se_diff = np.sqrt(p_A * (1 - p_A) / n_A + p_B * (1 - p_B) / n_B)
z_star = stats.norm.ppf(0.975)
ci_diff = (diff - z_star * se_diff, diff + z_star * se_diff)

print('=== A/B Test Results ===')
print(f'Variant A: {data_A.sum()}/{n_A} = {p_A:.1%}   95% CI: [{ci_A[0]:.1%}, {ci_A[1]:.1%}]')
print(f'Variant B: {data_B.sum()}/{n_B} = {p_B:.1%}   95% CI: [{ci_B[0]:.1%}, {ci_B[1]:.1%}]')
print(f'Difference (B−A): {diff:+.1%}  95% CI: [{ci_diff[0]:+.1%}, {ci_diff[1]:+.1%}]')
print(f'Does CI on difference exclude 0? {not (ci_diff[0] <= 0 <= ci_diff[1])}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: individual CIs ---
ax = axes[0]
variants  = ['Variant A', 'Variant B']
estimates = [p_A, p_B]
cis       = [ci_A, ci_B]
true_vals = [true_rate_A, true_rate_B]
colors    = ['steelblue', 'darkorange']

for i, (v, est, ci, tv, c) in enumerate(zip(variants, estimates, cis, true_vals, colors)):
    ax.errorbar(i, est, yerr=[[est - ci[0]], [ci[1] - est]],
                fmt='o', color=c, ms=10, capsize=8, capthick=2, lw=2, label=v)
    ax.scatter(i, tv, marker='*', s=180, color='black', zorder=5)

ax.set_xticks([0, 1])
ax.set_xticklabels(variants)
ax.set_ylabel('Conversion rate')
ax.set_title('Conversion Rates with 95% CIs\n(★ = true rate)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.legend()

# --- Right: CI on the difference ---
ax = axes[1]
ax.errorbar(0, diff, yerr=[[diff - ci_diff[0]], [ci_diff[1] - diff]],
            fmt='D', color='purple', ms=10, capsize=10, capthick=2, lw=2.5)
ax.axhline(0, color='crimson', lw=2, linestyle='--', label='No effect (Δ = 0)')
ax.axhline(true_rate_B - true_rate_A, color='black', lw=1.5,
           linestyle=':', label=f'True Δ = {true_rate_B - true_rate_A:+.1%}')

# Shade the CI region
ax.axhspan(ci_diff[0], ci_diff[1], alpha=0.15, color='purple')

ax.set_xticks([])
ax.set_ylabel('Difference in conversion (B − A)')
ax.set_title('95% CI on Difference (B − A)\nExcludes 0 → statistically significant')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:+.1%}'))
ax.legend()

plt.suptitle('A/B Test: Is Variant B Better?', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Reading the result

- If the CI on the **difference** excludes 0: the difference is statistically significant at the corresponding level.
- If it includes 0: we cannot rule out that B and A perform the same.
- Always check **practical significance** too — a statistically significant difference of 0.01% is rarely worth acting on.

---
## Recap

| Concept | Key point |
|---|---|
| **What a CI is** | A range computed from sample data that captures the true parameter at a given frequency |
| **Correct interpretation** | The *procedure* produces intervals containing μ X% of the time — not a probability for a single interval |
| **Width drivers** | Larger n → narrower; higher confidence → wider |
| **t vs z** | Use t when σ is unknown (almost always); t → z as n grows |
| **CI ≠ prediction interval** | CIs are about the *mean*, not individual observations |
| **Significance test link** | A 95% CI excludes a null value ⟺ a two-sided test at α=0.05 rejects that null |